## **Query PubChem Database for Approved Small Molecule Drugs**

ChEMBL has been my go-to chemical database. This is mostly due to the manual curation and user-friendly interface. These data quality features, one could argue, come at a data quanity expense, i.e., it takes a lot of time, effort, and funding to maitain this public database. PubChem is contrast has a quantity problem, which makes it a beast to tame. Until now my interactions have largely been to use PubChem to look at a single molecule to determine what is know. The NLM at NIH have done a huge service to curate and classify chemicals.

In [1]:
# Import modules
import time
import requests
import pandas as pd

# Expand to see all columns
pd.set_option("display.max_columns", None)

# Print versions
print(f"Pandas Version: {pd.__version__}")

# PubChem Good Citizen 
BATCH_SIZE = 100  # CIDs per PUG REST property request
RATE_DELAY = 0.25 # seconds between requests (≤5 req/s per PubChem policy)
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (compatible; PubChemApprovedCrawl/1.0; "
        "+https://pubchem.ncbi.nlm.nih.gov)"
    )
}

Pandas Version: 3.0.2


In [2]:
hnid = {}
# Chemicals in PubChem from Regulatory Resources
# https://pubchem.ncbi.nlm.nih.gov/classification/#hid=135
hnid["US_FDA_Class"] = "17060429" # US FDA Classification, 5052 Cpds
hnid["EU_EMA_Class"] = "17060390" # EU EMA Classification, 5353 Cpds
hnid["Japan_PDMA_Class"] = "17060437" # Japan PDMA Classification, 208 Cpds

# PubChem Compound TOC
# https://pubchem.ncbi.nlm.nih.gov/classification/#hid=72
hnid["US_FDA_Approved_Drugs"] = "5614020" # FDA Approved Drugs, 2563 Cpds
hnid["US_FDA_Orange_Book"] = "1857316" # FDA Orange Book, 2472 Cpds
hnid["EU_EMA_Drug_Info"] = "3647577" # EMA Drug Information, 5265 Cpds
hnid["Japan_PDMA_Info"] = "13313400" # Japan PDMA, 204 Cpds

cids = []
sources = []
for key, id in hnid.items():
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/classification/hnid/{id}/cids/JSON"
    response = requests.get(url)
    if response.ok:
        ids = response.json()["IdentifierList"]["CID"]
        cids.extend(ids)
        sources.extend([key] * len(ids))
    time.sleep(RATE_DELAY)

print(f"CIDs retrieved: {len(cids)}")
print(f"First 10: {cids[:10]}")

CIDs retrieved: 21032
First 10: [4, 11, 174, 176, 177, 180, 196, 206, 222, 223]


In [3]:
property_str = "SMILES,Title"
base_url   = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{}/property/{}/JSON"

records = []
for ii in range(0, len(cids), BATCH_SIZE):
    batch = cids[ii : ii + BATCH_SIZE]
    source_batch = sources[ii : ii + BATCH_SIZE]
    cid_str = ",".join(str(c) for c in batch)
    response = requests.get(base_url.format(cid_str, property_str))
    if response.ok:
        records.extend(response.json()["PropertyTable"]["Properties"])
    time.sleep(RATE_DELAY)
for ii in range(len(cids)):
    records[ii]["Source"] = sources[ii]
pubchem_df = pd.DataFrame(records)
pubchem_df = pubchem_df.groupby("CID").agg(
    SMILES=("SMILES", "first"),
    Title=("Title", "first"),
    Source=("Source", ",".join)
)
pubchem_df = pubchem_df.reset_index()
print(pubchem_df.shape)
pubchem_df.head()

(7934, 4)


,CID,SMILES,Title,Source
0,4,CC(CN)O,1-Amino-2-propanol,US_FDA_Class
1,11,C(CCl)Cl,"1,2-Dichloroethane",US_FDA_Class
2,51,C(CC(=O)O)C(=O)C(=O)O,2-Oxoglutaric Acid,"EU_EMA_Class,EU_EMA_Drug_Info"
3,135,C1=CC(=CC=C1C(=O)O)O,4-Hydroxybenzoic Acid,"EU_EMA_Class,EU_EMA_Drug_Info"
4,137,C(CC(=O)O)C(=O)CN,5-Aminolevulinic Acid,"EU_EMA_Class,EU_EMA_Drug_Info"


In [4]:
pubchem_df.to_csv("pubchem_approved_small_molecule_drugs.csv", index=False)

In [5]:
hnid = {}
# PubChem Compound TOC (Clinical trial & cancer drugs)
# https://pubchem.ncbi.nlm.nih.gov/classification/#hid=72
hnid["Clinical_Trials"] = "1856916" # Clinical Trials - US, EU, & Japan, 11879 Cpds
hnid["Cancer_Drugs"] = "5562093" # Cancer Drugs, 255 Cpds

# NCI Thesaurus (NCIt) NCI Thesaurus
# https://pubchem.ncbi.nlm.nih.gov/classification/#hid=112
hnid["NCI_Thesaurus"] = "4644188" # NCI Thesaurus, 19925 Cpds

cids = []
sources = []
for key, id in hnid.items():
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/classification/hnid/{id}/cids/JSON"
    response = requests.get(url)
    if response.ok:
        ids = response.json()["IdentifierList"]["CID"]
        cids.extend(ids)
        sources.extend([key] * len(ids))
    time.sleep(RATE_DELAY)

print(f"CIDs retrieved: {len(cids)}")
print(f"First 10: {cids[:10]}")

CIDs retrieved: 32144
First 10: [1, 2, 6, 38, 51, 119, 137, 144, 174, 175]


In [6]:
property_str = "SMILES,Title"
base_url   = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{}/property/{}/JSON"

records = []
for ii in range(0, len(cids), BATCH_SIZE):
    batch = cids[ii : ii + BATCH_SIZE]
    source_batch = sources[ii : ii + BATCH_SIZE]
    cid_str = ",".join(str(c) for c in batch)
    response = requests.get(base_url.format(cid_str, property_str))
    if response.ok:
        records.extend(response.json()["PropertyTable"]["Properties"])
    time.sleep(RATE_DELAY)
for ii in range(len(cids)):
    records[ii]["Source"] = sources[ii]
pubchem_df = pd.DataFrame(records)
pubchem_df = pubchem_df.groupby("CID").agg(
    SMILES=("SMILES", "first"),
    Title=("Title", "first"),
    Source=("Source", ",".join)
)
pubchem_df = pubchem_df.reset_index()
print(pubchem_df.shape)
pubchem_df.head()

(24828, 4)


,CID,SMILES,Title,Source
0,1,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,Acetyl-DL-carnitine,Clinical_Trials
1,2,CC(=O)OC(CC(=O)O)C[N+](C)(C)C,"2-(Acetyloxy)-3-carboxy-N,N,N-trimethylpropan-...",Clinical_Trials
2,6,C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl,"1-Chloro-2,4-Dinitrobenzene","Clinical_Trials,NCI_Thesaurus"
3,11,C(CCl)Cl,"1,2-Dichloroethane",NCI_Thesaurus
4,34,C(CCl)O,2-Chloroethanol,NCI_Thesaurus


In [7]:
pubchem_df.to_csv("pubchem_clinical_nci_small_molecule_drugs.csv", index=False)

In [8]:
# ATC Classification, 5678 Molecules
page = 1
total_pages = 2
base_url = (
    "https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/annotations/heading/JSON/"
    "?source=WHO%20Anatomical%20Therapeutic%20Chemical%20(ATC)%20Classification"
    "&heading_type=Compound"
    "&heading=ATC%20Code"
    "&response_type=display"
)
records = []
while page <= total_pages:
    url = f"{base_url}&page={page}"
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    if response.ok:
        data = response.json()
        total_pages = data.get("Annotations", {}).get("TotalPages", 1)
        annotations = data.get("Annotations", {}).get("Annotation", [])
        for ann in annotations:
            name = ann.get("Name", "")
            atc_code = ann.get("SourceID", "")
            atc_url = ann.get("URL", "")
            cids = ann.get("LinkedRecords", {}).get("CID", [])
            cid = ""
            if cids:
                cid = cids[0]

            records.append({
                "cid": cid,
                "Title": name,
                "atc_code": atc_code,
                "atc_url": atc_url,
            })
    page += 1
    time.sleep(RATE_DELAY)

In [9]:
property_str = "SMILES"
base_url   = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{}/property/{}/JSON"

rows = []
cids = [(ii, rr["cid"]) for ii, rr in enumerate(records) if rr["cid"] != ""]
for ii in range(0, len(cids), BATCH_SIZE):
    batch = cids[ii : ii + BATCH_SIZE]
    cid_str = ",".join(str(cc[1]) for cc in batch if cc[1] != "")
    response = requests.get(base_url.format(cid_str, property_str))
    response.raise_for_status()
    if response.ok:
        data_batch = response.json()
        rows.extend(data_batch.get("PropertyTable", {}).get("Properties", []))
    time.sleep(RATE_DELAY)
for ii, cc in enumerate(cids):
    records[cc[0]]["SMILES"] = rows[ii]["SMILES"]
pubchem_df = pd.DataFrame(records)
pubchem_df = pubchem_df.loc[:, ["cid", "SMILES", "Title", "atc_code", "atc_url"]]
print(pubchem_df.shape)
pubchem_df.head()

(5678, 5)


,cid,SMILES,Title,atc_code,atc_url
0,,NaN,Adalimumab,L04AB04,https://atcddd.fhi.no/atc_ddd_index/?code=L04AB04
1,49803313,CCC1=C(N=C(C(=N1)C(=O)N)NC2=CC(=C(C=C2)N3CCC(C...,Gilteritinib,L01EX13,https://atcddd.fhi.no/atc_ddd_index/?code=L01EX13
2,68706,C1=CC=C(C=C1)N2C=C(C(=N2)C3=CC=C(C=C3)Cl)CC(=O)O,Lonazolac,M01AB09,https://atcddd.fhi.no/atc_ddd_index/?code=M01AB09
3,168009,C(CN(CCO[N+](=O)[O-])CCO[N+](=O)[O-])N(CCO[N+]...,Tenitramine,C01DA38,https://atcddd.fhi.no/atc_ddd_index/?code=C01DA38
4,,NaN,Bermekimab,L01FX11,https://atcddd.fhi.no/atc_ddd_index/?code=L01FX11


In [10]:
pubchem_df.to_csv("pubchem_who_atc_classification_molecules.csv", index=False)